# Atividade I — Cálculo Numérico
## Erros numéricos em biofluidodinâmica

**Profa. Raquel Jahara Lobosco**

Integrantes e divisão de tarefas:

- (Felipe Marinho de Figueiredo) -
- (Bruno de Carvalho Morais) -


## Preparação

Importamos as bibliotecas e criamos as duas funções de apoio: uma para truncar e outra para calcular os erros.

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from matplotlib.patches import Circle

In [ ]:
# Trunca um número
def truncar(numero, casas):
    fator = 10 ** casas
    return math.trunc(numero * fator) / fator

# Calcula erro absoluto, relativo e percentual.
def calcular_erros(referencia, aproximacao):
    erro_abs = abs(referencia - aproximacao)
    if referencia == 0:
        erro_rel = float("nan")
    else:
        erro_rel = erro_abs / abs(referencia)
    erro_perc = 100 * erro_rel
    return erro_abs, erro_rel, erro_perc

## Parte A — Truncamento e arredondamento

Valores: x1 = 3,14159265, x2 = 98,76543, x3 = 0,00498765.
Para cada um usamos n = 0, 1, 2, 3, 4 casas e calculamos os erros.

In [ ]:
valores = {"x1": 3.14159265, "x2": 98.76543, "x3": 0.00498765}

print("valor | n | truncado | arredondado | Ea(trunc) | Ea(arred)")
for nome, x in valores.items():
    for n in range(5):
        xt = truncar(x, n)
        xr = round(x, n)
        ea_t, er_t, ep_t = calcular_erros(x, xt)
        ea_a, er_a, ep_a = calcular_erros(x, xr)
        print(f"{nome} | {n} | {xt:.5f} | {xr:.5f} | {ea_t:.2e} | {ea_a:.2e}")

### Gráfico do erro absoluto em função de n (escala log)

In [ ]:
for nome, x in valores.items():
    ns = range(5)
    ea_trunc = [calcular_erros(x, truncar(x, n))[0] for n in ns]
    ea_arred = [calcular_erros(x, round(x, n))[0] for n in ns]
    plt.semilogy(ns, ea_trunc, "o-", label=f"{nome} truncamento")
    plt.semilogy(ns, ea_arred, "s--", label=f"{nome} arredondamento")

plt.xlabel("n (casas decimais)")
plt.ylabel("erro absoluto")
plt.title("Erro absoluto x n")
plt.legend(fontsize=8)
plt.show()

**Arredondar sempre gera erro menor ou igual ao truncamento?**

Sim, menor ou igual. Olhando a tabela, o erro do arredondamento nunca é maior que o do truncamento. Às vezes eles são iguais (quando o algarismo cortado é menor que 5) e às vezes o arredondamento é melhor (quando é 5 ou mais). Isso acontece porque o arredondamento escolhe o número mais próximo, enquanto o truncamento sempre corta para baixo.

## Parte B — Medidas de pressão arterial

Dez medidas em mmHg. A referência é a média usando todos os algarismos.
Criamos quatro séries e calculamos as estatísticas de cada uma.

In [ ]:
med = [121.7, 120.4, 122.1, 119.8, 121.2, 120.9, 122.4, 121.0, 120.2, 121.5]

originais   = np.array(med)
arred_int   = np.array([round(v) for v in med])
truncados   = np.array([truncar(v, 0) for v in med])
arred_1casa = np.array([round(v, 1) for v in med])

series = {
    "originais": originais,
    "arred inteiro": arred_int,
    "truncado inteiro": truncados,
    "arred 1 casa": arred_1casa,
}

media_ref = originais.mean()
print(f"Média de referência: {media_ref:.4f} mmHg")
print()
print("série | média(mmHg) | mín(mmHg) | máx(mmHg) | amplitude(mmHg) | desvio(mmHg) | Ea(mmHg) | E%")
for nome, s in series.items():
    media = s.mean()
    minimo = s.min()
    maximo = s.max()
    amplitude = maximo - minimo
    desvio = s.std(ddof=1)   # desvio-padrão amostral
    ea, er, ep = calcular_erros(media_ref, media)
    print(f"{nome} | {media:.4f} | {minimo:.1f} | {maximo:.1f} | {amplitude:.1f} | {desvio:.4f} | {ea:.4f} | {ep:.4f}")

### Gráficos

In [ ]:
# 1) as dez medições nas quatro séries
x = range(1, 11)
for nome, s in series.items():
    plt.plot(x, s, "o-", label=nome)
plt.xlabel("medição")
plt.ylabel("pressão (mmHg)")
plt.title("Dez medições")
plt.legend(fontsize=8)
plt.show()

In [ ]:
# 2) e 3) erros absolutos e percentuais das médias
nomes = list(series.keys())
ea_lista = [calcular_erros(media_ref, series[n].mean())[0] for n in nomes]
ep_lista = [calcular_erros(media_ref, series[n].mean())[2] for n in nomes]

plt.bar(nomes, ea_lista)
plt.ylabel("erro absoluto (mmHg)")
plt.title("Erro absoluto da média")
plt.xticks(rotation=20)
plt.show()

plt.bar(nomes, ep_lista)
plt.ylabel("erro percentual (%)")
plt.title("Erro percentual da média")
plt.xticks(rotation=20)
plt.show()

In [ ]:
# 4) histograma dos originais e dos inteiros arredondados
plt.hist(originais, bins=6, alpha=0.6, label="originais")
plt.hist(arred_int, bins=6, alpha=0.6, label="inteiros arred")
plt.xlabel("pressão (mmHg)")
plt.ylabel("frequência")
plt.title("Histograma")
plt.legend()
plt.show()

**A representação inteira preserva a média e a variabilidade?**

Arredondar para inteiro mantém a média bem próxima da referência, porque os arredondamentos para cima e para baixo quase se cancelam. Truncar para inteiro puxa tudo para baixo e afasta mais a média. Nos dois casos inteiros a variabilidade fica um pouco distorcida, porque a diferença real entre as medidas é de décimos e o inteiro perde essa informação.


## Parte C — Vazão em tubo pequeno (Hagen–Poiseuille)

$$Q = \frac{\pi r^4 \Delta p}{8 \mu L}$$

Com r = 0,80 mm, L = 0,20 m, µ = 3,5e-3 Pa·s, Δp = 1200 Pa.

In [ ]:
# Fórmula de Hagen-Poiseuille
def vazao(r, dp, mu, L):
    return math.pi * r**4 * dp / (8 * mu * L)

r  = 0.80e-3    # m
L  = 0.20       # m
mu = 3.5e-3     # Pa.s
dp = 1200       # Pa

Q_ref = vazao(r, dp, mu, L)
print(f"Q de referência = {Q_ref:.4e} m^3/s")

### Precisão das variáveis

In [ ]:
r_mm = 0.80

# aproximações de cada variável
r_ar = round(r_mm, 1) * 1e-3
r_tr = truncar(r_mm, 1) * 1e-3
mu_ar = round(mu, 3)
mu_tr = truncar(mu, 3)
dp_ar = round(dp / 100) * 100
dp_tr = truncar(dp / 100, 0) * 100

cenarios = {
    "1) todos originais": vazao(r, dp, mu, L),
    "2) r arred":         vazao(r_ar, dp, mu, L),
    "2) r trunc":         vazao(r_tr, dp, mu, L),
    "3) mu arred":        vazao(r, dp, mu_ar, L),
    "3) mu trunc":        vazao(r, dp, mu_tr, L),
    "4) dp arred":        vazao(r, dp_ar, mu, L),
    "4) dp trunc":        vazao(r, dp_tr, mu, L),
    "5) todas arred":     vazao(r_ar, dp_ar, mu_ar, L),
    "5) todas trunc":     vazao(r_tr, dp_tr, mu_tr, L),
}

print("cenário | Q(m^3/s) | Ea(m^3/s) | E%")
for nome, q in cenarios.items():
    ea, er, ep = calcular_erros(Q_ref, q)
    print(f"{nome} | {q:.4e} | {ea:.2e} | {ep:.4f}")

> Alguns cenários dão erro zero porque a aproximação não muda o valor: 0,80 arredondado ou truncado em 1 casa continua 0,80, e 1200 em centenas continua 1200. Quem muda de verdade é a viscosidade (0,0035 vira 0,004 arredondando e 0,003 truncando), por isso ela gera os maiores erros.

### Sensibilidade ao raio (0,70 a 0,90 mm)

In [ ]:
raios = np.linspace(0.70, 0.90, 21)

Q_r  = np.array([vazao(rr*1e-3, dp, mu, L) for rr in raios])
Q_ar = np.array([vazao(round(rr,1)*1e-3, dp, mu, L) for rr in raios])
Q_tr = np.array([vazao(truncar(rr,1)*1e-3, dp, mu, L) for rr in raios])

ea_ar = np.abs(Q_r - Q_ar)
er_ar = ea_ar / Q_r

# gráfico 1: Q em função de r
plt.plot(raios, Q_r, label="Q referência")
plt.plot(raios, Q_ar, "--", label="r arredondado")
plt.xlabel("r (mm)"); plt.ylabel("Q (m^3/s)")
plt.title("Q em função de r"); plt.legend(); plt.show()

# gráfico 2: erro absoluto
plt.plot(raios, ea_ar)
plt.xlabel("r (mm)"); plt.ylabel("erro absoluto (m^3/s)")
plt.title("Erro absoluto de Q"); plt.show()

# gráfico 3: erro relativo
plt.plot(raios, er_ar)
plt.xlabel("r (mm)"); plt.ylabel("erro relativo")
plt.title("Erro relativo de Q"); plt.show()

# gráfico 4: erro percentual
plt.plot(raios, 100*er_ar)
plt.xlabel("r (mm)"); plt.ylabel("erro percentual (%)")
plt.title("Erro percentual de Q"); plt.show()

**Por que o erro no raio é amplificado na vazão?**

Porque Q depende de r elevado à quarta potência. Quando o raio tem um erro relativo pequeno, esse erro aparece multiplicado por 4 na vazão (dQ/Q ≈ 4·dr/r). Então um erro de 1% no raio vira cerca de 4% na vazão. O expoente 4 é o motivo da amplificação.